# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JunaidAhamed-7777/flyrank-machine-learning/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

Audit capstone metrics the way the March 2026 FlyRank paper was audited: methodology questions on published findings, then grouped/time validation on our Week-5 model.


## 1. Two paper findings + my methodology questions

*Source: `docs/flyrank-seo-research-march-2026.pdf` (text extracted with pypdf; readable).*

**Finding A (Finding #1 — The Anatomy of Growing Content).** In the portfolio comparison of rising versus falling impression trends, declining pages average about **2.3K words** and **230 days** of age versus **3.2K words** and **184 days** for growing pages — the paper frames this as older, thinner URLs losing momentum while still carrying impressions.

**Methodology question:** *Does the “up vs down” label use the same 30-day-versus-previous-30-day impression window as the operational refresh queue, and are word count and age measured strictly before that label window closes?* That matters because age and length are often updated after a refresh; if metadata or body length changed during the outcome window, the comparison mixes pre-decision structure with post-change content and overstates how much “thin/older” *precedes* decline.

**Finding B (Finding #3 — Click Capture by Position Tier).** The paper reports portfolio **weighted CTR** falling from about **0.423%** in the top-3 tier to **0.050%** in the deep tier, arguing that click capture compresses quickly as average position worsens.

**Methodology question:** *Are these tier CTRs computed from pooled clicks/impressions at portfolio scope, and does any holdout or appendix model reuse tier-level aggregates that were fit on the same pages being ranked?* Portfolio-level weighted CTR is a useful descriptive number; if a prioritisation model or subgroup claim reuses tier medians computed on the evaluation set, the validation design no longer supports generalisation to unseen sites or clients.

In [1]:
from pathlib import Path
import pypdf
pdf = Path('../../docs/flyrank-seo-research-march-2026.pdf')
text = ''.join((p.extract_text() or '') for p in pypdf.PdfReader(pdf).pages)
print(f'PDF readable: {len(text) > 10000} ({len(text)} chars)')
print('Sample:', text[500:900].replace('\n', ' ')[:200])


PDF readable: True (53356 chars)
Sample: 79,632 search impressions, 1,514,819  clicks,1,635,404 sessions, and 17,344 AI sessions. Rather than reproducing dashboard screenshots or internal table views, we tested portfolio-level questions abou


## 2. My model under an honest split (before/after)

*Re-run Week-5 RF: w05 holdout (before), LOO-client + time-aware (after). Precision@50 only; ranking within each evaluation frame.*


In [2]:

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "work" / "notebooks"
REPO_ROOT = NOTEBOOK_DIR.parents[1]
OUT_DIR = REPO_ROOT / "work" / "outputs"

DECISION_DATE_MAR = "2026-03-31"
DECISION_DATE_FEB = "2026-02-28"
RANDOM_STATE = 42
FEATURES_HONEST = ["imp_last30", "avg_pos_last30", "ctr_last30", "content_age_days"]
FEATURES_LEAKY = FEATURES_HONEST + ["log_imp_prev30"]

TIER_WEIGHT = {
    "top_3": 1.0,
    "page_1": 0.9,
    "striking": 0.85,
    "page_3_5": 0.7,
    "deep": 0.5,
    "no_data": 0.0,
}


def position_tier(pos: float) -> str:
    if pd.isna(pos) or pos <= 0:
        return "no_data"
    if pos <= 3:
        return "top_3"
    if pos <= 10:
        return "page_1"
    if pos <= 20:
        return "striking"
    if pos <= 50:
        return "page_3_5"
    return "deep"


def precision_at_k(y_true, scores, k: int) -> float:
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


def w04_baseline_score(pages: pd.DataFrame) -> pd.Series:
    tier_median_ctr = pages.groupby("position_tier")["ctr_last30"].transform("median")
    ctr_headroom = (tier_median_ctr - pages["ctr_last30"]).clip(lower=0)
    tier_weight = pages["position_tier"].map(TIER_WEIGHT).fillna(0.0)
    return pages["impressions_90d"] * ctr_headroom * tier_weight


def make_rf():
    return RandomForestClassifier(
        class_weight="balanced_subsample",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def load_pages(decision_date: str, month_globs: list[str]) -> pd.DataFrame:
    import duckdb

    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        local = REPO_ROOT / "work" / ".hf_token_local"
        if local.is_file():
            hf_token = local.read_text(encoding="utf-8").strip()
    if not hf_token:
        raise RuntimeError("HF_TOKEN missing")

    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"
    dim_content = f"read_parquet('{rel}/dim_content.parquet')"
    paths = ",\n        ".join(
        f"'{rel}/fact_content_daily_performance/{m}/*.parquet'" for m in month_globs
    )
    fact = f"read_parquet([{paths}])"
    sql = f"""
    WITH daily AS (
        SELECT * FROM {fact}
        WHERE report_date <= DATE '{decision_date}'
    ),
    agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date > DATE '{decision_date}' - INTERVAL 90 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS impressions_90d,
            SUM(CASE WHEN report_date > DATE '{decision_date}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date > DATE '{decision_date}' - INTERVAL 60 DAY
                      AND report_date <= DATE '{decision_date}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_impressions, 0) ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date > DATE '{decision_date}' - INTERVAL 30 DAY
                     THEN COALESCE(gsc_clicks, 0) ELSE 0 END) AS clk_last30,
            AVG(CASE WHEN report_date > DATE '{decision_date}' - INTERVAL 30 DAY
                     THEN gsc_avg_position END) AS avg_pos_last30,
            BOOL_AND(gsc_data_available IS TRUE) AS gsc_ok
        FROM daily
        GROUP BY 1, 2
    ),
    labeled AS (
        SELECT
            a.*,
            CASE
                WHEN a.imp_prev30 = 0 AND a.imp_last30 > 0 THEN 'new'
                WHEN a.imp_prev30 = 0 AND a.imp_last30 = 0 THEN 'flat'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 > 20 THEN 'up'
                WHEN a.imp_prev30 > 0 AND (a.imp_last30 - a.imp_prev30) * 100.0 / a.imp_prev30 < -20 THEN 'down'
                ELSE 'stable'
            END AS trend_direction
        FROM agg a
        WHERE a.imp_last30 > 0 AND a.gsc_ok IS TRUE
    )
    SELECT
        l.client_hash_id AS client_id,
        l.content_hash_id AS content_id,
        l.impressions_90d,
        l.imp_last30,
        l.imp_prev30,
        l.clk_last30,
        l.avg_pos_last30,
        l.trend_direction,
        DATE_DIFF('day', d.content_created_date, DATE '{decision_date}') AS content_age_days
    FROM labeled l
    INNER JOIN {dim_content} d USING (content_hash_id)
    WHERE d.content_created_date <= DATE '{decision_date}'
      AND DATE_DIFF('day', d.content_created_date, DATE '{decision_date}') >= 90
    """
    return con.sql(sql).df()


def enrich(pages: pd.DataFrame) -> pd.DataFrame:
    pages = pages.copy()
    pages["ctr_last30"] = np.where(
        pages["imp_last30"] > 0,
        pages["clk_last30"] / pages["imp_last30"] * 100.0,
        0.0,
    )
    pages["log_imp_prev30"] = np.log1p(pages["imp_prev30"])
    pages["position_tier"] = pages["avg_pos_last30"].map(position_tier)
    pages["is_declining_label"] = pages["trend_direction"].str.lower().eq("down").astype(int)
    pages["baseline_score"] = w04_baseline_score(pages)
    return pages


def make_client_holdout(frame: pd.DataFrame):
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = np.array(sorted(client_series.unique()))
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(unique_clients)
    test_n = max(1, int(round(len(shuffled) * 0.2)))
    test_clients = set(shuffled[:test_n])
    test_mask = client_series.isin(test_clients).to_numpy()
    train_idx = np.where(~test_mask)[0]
    test_idx = np.where(test_mask)[0]
    return train_idx, test_idx, test_clients


print("Loading March 2026 decision slice...")
pages = enrich(
    load_pages(
        DECISION_DATE_MAR,
        ["month=2026-01", "month=2026-02", "month=2026-03"],
    )
)
y = pages["is_declining_label"]
X = pages[FEATURES_HONEST].replace([np.inf, -np.inf], np.nan).fillna(0)

# --- Before: w05 single holdout ---
train_idx, test_idx, holdout_clients = make_client_holdout(pages)
rf = make_rf()
rf.fit(X.iloc[train_idx], y.iloc[train_idx])
rf_probs_holdout = rf.predict_proba(X.iloc[test_idx])[:, 1]
p50_w05 = precision_at_k(y.iloc[test_idx], rf_probs_holdout, 50)
p50_w05_baseline = precision_at_k(
    y.iloc[test_idx], pages.iloc[test_idx]["baseline_score"], 50
)

# In-sample reference (same March frame)
rf_in = make_rf()
rf_in.fit(X, y)
p50_insample = precision_at_k(y, rf_in.predict_proba(X)[:, 1], 50)

# --- After #1: LOO-client (P@50 within each held-out client's pages) ---
loo_rf, loo_bl = [], []
for cid in sorted(pages["client_id"].unique()):
    tr = pages["client_id"] != cid
    te = ~tr
    y_tr, y_te = y[tr], y[te]
    if y_tr.nunique() < 2 or te.sum() == 0:
        continue
    m = make_rf()
    m.fit(X[tr], y_tr)
    loo_rf.append(precision_at_k(y_te, m.predict_proba(X[te])[:, 1], 50))
    loo_bl.append(precision_at_k(y_te, pages.loc[te, "baseline_score"], 50))

loo_mean = float(np.mean(loo_rf))
loo_std = float(np.std(loo_rf, ddof=0))
loo_zeros = sum(1 for p in loo_rf if p == 0.0)

# --- After #2: time-aware (train Feb, score March) ---
pages_feb = enrich(load_pages(DECISION_DATE_FEB, ["month=2026-01", "month=2026-02"]))
X_feb = pages_feb[FEATURES_HONEST].replace([np.inf, -np.inf], np.nan).fillna(0)
y_feb = pages_feb["is_declining_label"]
rf_time = make_rf()
rf_time.fit(X_feb, y_feb)
p50_time = precision_at_k(y, rf_time.predict_proba(X)[:, 1], 50)

global_baseline = precision_at_k(y, pages["baseline_score"], 50)

rows = [
    ("In-sample (w05 reference)", p50_insample, "RF trained and ranked on full March slice — calibration only"),
    ("w05 single client-holdout (22/6)", p50_w05, f"Seed {RANDOM_STATE}; {len(holdout_clients)} clients held out"),
    ("Leave-one-client-out (mean ± std)", f"{loo_mean:.3f} ± {loo_std:.3f}", f"P@50 within each client; {loo_zeros} folds at 0.0"),
    ("Time-aware (train Feb, test Mar)", p50_time, "Train on Feb decision slice; rank all March pages"),
    ("w04 baseline (full slice)", global_baseline, "Reference only"),
    ("w04 baseline (same split as w05 holdout)", p50_w05_baseline, "Apples-to-apples vs w05 RF holdout"),
    ("w04 baseline (LOO-client mean)", float(np.mean(loo_bl)), "Same LOO protocol as RF"),
]
table = pd.DataFrame(rows, columns=["Regime", "P@50", "Notes"])
print(table.to_string(index=False))

audit = {
    "p50_insample_rf": p50_insample,
    "p50_w05_holdout_rf": p50_w05,
    "p50_w05_holdout_baseline": p50_w05_baseline,
    "p50_loo_rf_mean": loo_mean,
    "p50_loo_rf_std": loo_std,
    "p50_time_aware_rf": p50_time,
    "global_baseline_p50": global_baseline,
    "loo_baseline_mean": float(np.mean(loo_bl)),
}
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "w06_audit_metrics.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
print(f"\nWrote {OUT_DIR / 'w06_audit_metrics.json'}")


Loading March 2026 decision slice...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                                  Regime          P@50                                                        Notes
               In-sample (w05 reference)          0.92 RF trained and ranked on full March slice — calibration only
        w05 single client-holdout (22/6)          0.38                                  Seed 42; 6 clients held out
       Leave-one-client-out (mean ± std) 0.258 ± 0.237                      P@50 within each client; 7 folds at 0.0
        Time-aware (train Feb, test Mar)          0.44            Train on Feb decision slice; rank all March pages
               w04 baseline (full slice)          0.34                                               Reference only
w04 baseline (same split as w05 holdout)          0.24                           Apples-to-apples vs w05 RF holdout
          w04 baseline (LOO-client mean)      0.173445                                      Same LOO protocol as RF

Wrote C:\aaFiles\aaGithub\FlyrankAI\flyrank-machine-learning\work\outpu

### Before/after readout

Re-running the w05 Random Forest (four honest features, same hyperparameters) yields **P@50 = 0.38** on the six-client holdout — not the **0.40** stated in w05 markdown (w05’s *executed* table showed **0.34**, so neither prose nor this re-run should be treated as a sealed number). Under **leave-one-client-out**, mean **P@50 ≈ 0.26 ± 0.24** across 28 clients (seven folds at **0.0** where the top-50 queue contained no declining pages). The single holdout was optimistic versus LOO; the time-aware check (**train February, test March**) measured **P@50 = 0.44** on the full March ranking frame — higher than LOO but still not a client-generalisation guarantee. For decision-support language: the grouped estimate sits near **0.26**, close to the **LOO baseline mean ≈ 0.17** and the global rule reference **0.34**, not a stable **+0.16** lift over the holdout baseline (**0.24**).

## 3. Leakage audit

### Part A — Per-feature audit (honest four-feature frame)

| Feature | How derived | Could it have been computed without the outcome? | Verdict |
|---|---|---|---|
| `imp_last30` | Sum of `gsc_impressions` where `report_date` ∈ (decision_date − 30d, decision_date] (w03/w05 SQL) | Yes — exposure in the pre-decision trailing window only | **safe** |
| `avg_pos_last30` | Mean `gsc_avg_position` over the same trailing 30 days | Yes — same window, no label columns | **safe** |
| `ctr_last30` | `clk_last30 / imp_last30 × 100` with clicks summed over that same trailing 30 days | Yes — clicks and impressions are co-terminous with the feature window, not the label ratio itself | **safe** |
| `content_age_days` | `DATE_DIFF(day, content_created_date, decision_date)` from `dim_content`, rows with created_date ≤ decision_date | Yes with caveat — created date is treated as immutable metadata; rows with created_date after decision are excluded in SQL | **safe with caveat** |

No feature marked **suspect**; nothing dropped and Section 2 not re-run.

### Part B — w05 leak already caught

`log_imp_prev30` remains **excluded** from `FEATURES_HONEST`. It is a monotone transform of `imp_prev30`, which enters the decline label \((imp_last30 - imp_prev30)/imp_prev30\). Re-training with it on the w05 holdout still yields **P@50 ≈ 1.00** versus **≈ 0.38** without it (see code output) — the same collapse pattern as w03’s `trend_pct` demo.

### Part C — Hidden leakage checks

- **`ctr_last30` / `avg_pos_last30` windows:** Warehouse query on March daily rows shows the trailing-30 window ends **2026-03-31** and starts **2026-03-02** (30 distinct days) — nothing after the decision date.
- **`content_age_days`:** Uses `content_created_date` joined from `dim_content` with `content_created_date <= decision_date`; export max date exceeds March but eligible rows are capped at the decision date (not snapshot-updated `content_updated_date`, which stays excluded).
- **`avg_pos_last30`:** Built only from daily rows with `report_date <= 2026-03-31` inside the last-30 filter — no post-decision days.

In [3]:

# Leakage demo + hidden window checks + error examples (Section 3 support)
import duckdb

X_leaky = pages[FEATURES_LEAKY].replace([np.inf, -np.inf], np.nan).fillna(0)
rf_leak = make_rf()
rf_leak.fit(X_leaky.iloc[train_idx], y.iloc[train_idx])
leaky_p50 = precision_at_k(y.iloc[test_idx], rf_leak.predict_proba(X_leaky.iloc[test_idx])[:, 1], 50)
print(f"log_imp_prev30 still excluded from honest frame: {FEATURES_HONEST}")
print(f"w05 reminder — RF P@50 WITH log_imp_prev30: {leaky_p50:.2f} | WITHOUT: {p50_w05:.2f}")

hf_token = os.environ.get("HF_TOKEN") or (REPO_ROOT / "work" / ".hf_token_local").read_text().strip()
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
win = con.sql(f"""
WITH daily AS (
  SELECT report_date FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
  WHERE report_date <= DATE '2026-03-31'
    AND report_date > DATE '2026-03-31' - INTERVAL 30 DAY
)
SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d, COUNT(DISTINCT report_date) AS days FROM daily
""").df()
print("\nTrailing-30 window for March decision (GSC daily):")
print(win.to_string(index=False))

perm = permutation_importance(
    rf, X.iloc[test_idx], y.iloc[test_idx], n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
)
imp = pd.Series(perm.importances_mean, index=FEATURES_HONEST).sort_values(ascending=False)
print("\nPermutation importance (RF, w05 holdout, honest frame):")
print(imp.to_string())

holdout = pages.iloc[test_idx].copy()
holdout["rf_score"] = rf_probs_holdout
top50 = holdout.nlargest(min(50, len(holdout)), "rf_score")
tp = int(top50["is_declining_label"].sum())
fp = int(len(top50) - tp)
print(f"\nHoldout top-50 error sketch: {tp} TP / {fp} FP (P@50={tp/len(top50):.2f})")
ranked = holdout.sort_values("rf_score", ascending=False).reset_index(drop=True)
first_fp = ranked[ranked["is_declining_label"] == 0].head(1)
if not first_fp.empty:
    r = int(first_fp.index[0]) + 1
    row = first_fp.iloc[0]
    print(
        f"First FP at rank {r}: tier={row['position_tier']} ctr_last30={row['ctr_last30']:.2f}% "
        f"imp_last30={row['imp_last30']:.0f}"
    )


log_imp_prev30 still excluded from honest frame: ['imp_last30', 'avg_pos_last30', 'ctr_last30', 'content_age_days']
w05 reminder — RF P@50 WITH log_imp_prev30: 1.00 | WITHOUT: 0.38


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Trailing-30 window for March decision (GSC daily):
     min_d      max_d  days
2026-03-02 2026-03-31    30



Permutation importance (RF, w05 holdout, honest frame):
imp_last30         -0.005067
ctr_last30         -0.006259
content_age_days   -0.009836
avg_pos_last30     -0.010879

Holdout top-50 error sketch: 19 TP / 31 FP (P@50=0.38)
First FP at rank 1: tier=striking ctr_last30=0.00% imp_last30=187
